In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

train_df = pd.read_csv('/content/drive/MyDrive/2026-1 시냅스 활동/train.csv')
test_df = pd.read_csv('/content/drive/MyDrive/2026-1 시냅스 활동/test.csv')

print(train_df.head())
print(test_df.head())

  file_name  label
0   001.PNG      9
1   002.PNG      4
2   003.PNG      1
3   004.PNG      1
4   005.PNG      6
  file_name
0   001.PNG
1   002.PNG
2   003.PNG
3   004.PNG
4   005.PNG


In [3]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']
)

print(f'학습 데이터: {len(train_data)}장')
print(f'검증 데이터: {len(val_data)}장')

학습 데이터: 578장
검증 데이터: 145장


In [4]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [5]:
from torch.utils.data import Dataset
from PIL import Image

class CustomDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['file_path']
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if 'label' in self.df.columns:
            label = self.df.iloc[idx]['label']
            return image, label

        return image

train_dataset = CustomDataset(train_data.reset_index(drop=True), transform=train_transform)
val_dataset = CustomDataset(val_data.reset_index(drop=True), transform=val_transform)
test_dataset = CustomDataset(test_df.reset_index(drop=True), transform=val_transform)

print(f'train dataset: {len(train_dataset)}장')
print(f'val dataset: {len(val_dataset)}장')
print(f'test dataset: {len(test_dataset)}장')

train dataset: 578장
val dataset: 145장
test dataset: 199장


In [6]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print(f'train_loader: {len(train_loader)}배치')
print(f'val_loader: {len(val_loader)}배치')
print(f'test_loader: {len(test_loader)}배치')

train_loader: 19배치
val_loader: 5배치
test_loader: 7배치
